# DLT Pipeline

In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql import DataFrame as df
from pyspark.sql.window import Window
from functools import reduce
from pyspark.sql.types import StructType, StructField, StringType, MapType

The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File /databricks/python/lib/python3.12/site-packages/pyspark/pipelines/__init__.py:42
     41 try:
---> 42     from dlt import *
     43 except ImportError:

ModuleNotFoundError: No module named 'dlt'

During handling of the above exception, another exception occurred:

TypeError                                 Traceback (most recent call last)
File <command-7133316255034260>, line 1
----> 1 from pyspark import pipelines as dp
      2 from pyspark.sql import functions as F
      3 from pyspark.sql import DataFrame as df

File /databricks/python/lib/python3.12/site-packages/pyspark/pipelines/__init__.py:44
     42         from dlt import *
     43     except ImportError:
---> 44         raise PySparkException(errorClass="PIPELINES_NOT_SUPPORTED")
     45 # END: EDGE

File /databricks/python/lib/python3.12/site-packages/pyspark/errors/exc

In [0]:
RAW = "/Volumes/alpha_vantage/bronze/bronze-landing"

In [0]:
raw_schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("timeseries", StructType([
        StructField("time_series_daily", 
            MapType(StringType(), MapType(StringType(), StringType())), 
            True
        )
    ]), True),
    StructField("quote", StructType([
        StructField("global_quote", 
            MapType(StringType(), MapType(StringType(), StringType())), 
            True
        )
    ]), True)
])

## Create Bronze table

In [0]:
symbols  = ["AAPL", "FGI", "GOOG", "MSFT"]

def create_bronze_table(ticker):
    @dp.table(name=f"alpha_vantage.bronze.{ticker}_daily_bronze")
    def daily_bronze():
        return (
            spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "json")
            .schema(raw_schema)
            .load(f"{RAW}/{ticker}/")
        )
    @dp.table(name=f"jrvs_databricks_fundamentals.bronze.{ticker}_quote_bronze")
    def quote_bronze():
        return (
            spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "json")
            .schema(raw_schema)
            .load(f"{RAW}/{ticker}/")
        )

for symbol in symbols:
    create_bronze_table(symbol)

## Create Silver Table


In [0]:
def create_silver_table(ticker):
    @dp.view(name=f"{ticker}_daily_silver")
    def daily_silver_clean():
        return (
            spark.readStream.table(f"alpha_vantage.bronze.{ticker}_daily_bronze")
            .select(
                "symbol",
                F.explode(F.col("timeseries.`time_series_daily`")).alias("trade_date", "ohlcv")
            )
            .withColumn("trade_date", F.to_date("trade_date"))
            .withColumn("open",   F.round(F.col("ohlcv")["1. open"].cast("double"), 2))
            .withColumn("high",   F.round(F.col("ohlcv")["2. high"].cast("double"), 2))
            .withColumn("low",    F.round(F.col("ohlcv")["3. low"].cast("double"), 2))
            .withColumn("close",  F.round(F.col("ohlcv")["4. close"].cast("double"), 2))
            .withColumn("volume", F.col("ohlcv")["5. volume"].cast("long"))
            .drop("ohlcv")
        )

    dp.create_streaming_table(
    name=f"alpha_vantage.silver.{ticker}_daily_silver",
    expect_all_or_drop={
        "has_symbol": "symbol IS NOT NULL",
        "has_date": "trade_date IS NOT NULL",
        "close_positive": "close > 0",
    }
    )

    dp.apply_changes(
        target=f"alpha_vantage.silver.{ticker}_daily_silver",
        source=f"{ticker}_daily_silver",
        keys=["trade_date"],   
        ignore_null_updates=True,
        sequence_by=F.col("trade_date"),
        stored_as_scd_type=1          
    )

    @dp.view(name=f"{ticker}_quote_silver_clean")
    def quote_silver_clean():
        return (
            spark.readStream.table(f"alpha_vantage.bronze.{ticker}_quote_bronze")
                .withColumn("latest_trading_day", F.to_date("latest_trading_day"))
                .withColumn("price", F.round(F.col("price").cast("double"), 2))
                .withColumn("previous_close", F.round(F.col("previous_close").cast("double"), 2))
                .withColumn("price_change", F.round(F.col("price_change").cast("double"), 2))
                .withColumn("change_percent", F.round(F.col("change_percent").cast("double"), 2))
                .withColumn("volume", F.col("volume").cast("long"))
        )

    dp.create_streaming_table(
    name=f"alpha_vantage.silver.{ticker}_quote_silver",
    expect_all_or_drop={
        "has_symbol": "symbol IS NOT NULL",
        "has_date": "latest_trading_day IS NOT NULL",
    }
    )

    dp.apply_changes(
        target=f"alpha_vantage.silver.{ticker}_quote_silver",
        source=f"{ticker}_quote_silver_clean",
        keys=["latest_trading_day"],   
        ignore_null_updates=True,
        sequence_by=F.col("latest_trading_day"),
        stored_as_scd_type=1          
    )
for symbol in symbols:
    create_silver_table(symbol)


## Create Gold Table


In [0]:
def create_gold_table(name):
    @dp.table(name=f"alpha_vantage.gold.{name}_gold")
    def gold():

        daily = spark.read.table(f"alpha_vantage.silver.{name}_daily_silver").select("symbol", "trade_date", "open", "high", "low","close", "volume")
        quote = spark.read.table(f"alpha_vantage.silver.{name}_quote_silver").select("symbol", "change_percent")
        
        w = Window.orderBy("trade_date")

        out = daily.join(quote, "symbol")
        out = out.withColumnRenamed("close", "price")

        return out
for symbol in symbols:
    create_gold_table(symbol)